In [ ]:
!ls /content/Voyage-Analytics/data/raw

In [ ]:
import pandas as pd

# Load the datasets
users = pd.read_csv("/content/Voyage-Analytics/data/raw/users.csv")
hotels = pd.read_csv("/content/Voyage-Analytics/data/raw/hotels.csv")
flights = pd.read_csv("/content/Voyage-Analytics/data/raw/flights.csv")

# Check the first few rows of each dataset
print(users.head())
print(hotels.head())
print(flights.head())

In [ ]:
# Basic Info about each dataset
print("Users Dataset Info:")
print(users.info())

print("Hotels Dataset Info:")
print(hotels.info())

print("Flights Dataset Info:")
print(flights.info())

# Check for missing values
print("Missing Values in Users Dataset:")
print(users.isnull().sum())

print("Missing Values in Hotels Dataset:")
print(hotels.isnull().sum())

print("Missing Values in Flights Dataset:")
print(flights.isnull().sum())

# Checking the first few records in each dataset
print("First 5 records in Users Dataset:")
print(users.head())

print("First 5 records in Hotels Dataset:")
print(hotels.head())

print("First 5 records in Flights Dataset:")
print(flights.head())

In [ ]:
import pandas as pd

# Create a sample ratings DataFrame, assuming user interactions with hotels
ratings = pd.DataFrame({
    'user_id': [1, 1, 2, 2, 3, 3],    # User IDs
    'hotel_id': [101, 102, 101, 103, 102, 104],  # Hotel IDs
    'rating': [5, 4, 3, 5, 2, 4]    # Ratings (or interactions)
})

# Display the ratings DataFrame
print(ratings)

In [ ]:
# Create the user-item matrix (ratings matrix)
user_item_matrix = pd.pivot_table(ratings, values='rating', index='user_id', columns='hotel_id', fill_value=0)

# Display the user-item matrix
print(user_item_matrix)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Calculate cosine similarity between users
cosine_sim = cosine_similarity(user_item_matrix)

# Convert to DataFrame for better readability
cosine_sim_df = pd.DataFrame(cosine_sim, index=user_item_matrix.index, columns=user_item_matrix.index)

# Display the cosine similarity matrix
print(cosine_sim_df)

In [ ]:
def recommend_hotels(user_id, cosine_sim_df, user_item_matrix, top_n=5):
    # Get the most similar users to the given user
    similar_users = cosine_sim_df[user_id].sort_values(ascending=False)[1:]  # Exclude the user itself

    # Get the top N similar users
    top_users = similar_users.head(top_n).index

    # Get the hotels that these similar users interacted with
    recommended_hotels = user_item_matrix.loc[top_users].sum(axis=0)

    # Sort the hotels by the total interaction score and return the top N
    recommended_hotels = recommended_hotels.sort_values(ascending=False).head(top_n)

    return recommended_hotels

# Example usage for a user
user_id = 1  # Replace with the actual user_id
recommended_hotels = recommend_hotels(user_id, cosine_sim_df, user_item_matrix)
print(f"Recommended Hotels for User {user_id}:")
print(recommended_hotels)

In [ ]:
from scipy.sparse.linalg import svds
import numpy as np
import pandas as pd

# 1. Create sample ratings data
ratings = pd.DataFrame({
    'user_id': [1, 1, 2, 2, 3, 3],
    'hotel_id': [101, 102, 101, 103, 102, 104],
    'rating': [5, 4, 3, 5, 2, 4]
})

# 2. Create the user-item matrix
user_item_matrix = pd.pivot_table(ratings, values='rating', index='user_id', columns='hotel_id', fill_value=0)
pivot_df = user_item_matrix.copy()

# 3. Decompose the matrix using SVD
pivot_matrix = pivot_df.values
user_ratings_mean = np.mean(pivot_matrix, axis=1)
matrix_demeaned = pivot_matrix - user_ratings_mean.reshape(-1, 1)

# k=2 because our sample data is small
U, sigma, Vt = svds(matrix_demeaned, k=2)
sigma = np.diag(sigma)

# 4. Reconstruct the matrix to get predicted ratings
all_user_predicted_ratings = np.dot(np.dot(U, sigma), Vt) + user_ratings_mean.reshape(-1, 1)
preds_df = pd.DataFrame(all_user_predicted_ratings, columns=pivot_df.columns, index=pivot_df.index)

def recommend_svd(user_id, num_recommendations=5):
    user_row_number = user_id - 1
    sorted_user_predictions = preds_df.iloc[user_row_number].sort_values(ascending=False)
    return sorted_user_predictions.head(num_recommendations)

# Example usage
user_id = 1
predictions = recommend_svd(user_id)
print(f"SVD Predicted Ratings for User {user_id}:")
print(predictions)

In [ ]:
from sklearn.metrics import mean_squared_error
import numpy as np

# Assuming you have true ratings and predicted ratings
# Example: predicted_ratings and actual_ratings
predicted_ratings = [4.5, 3.0, 5.0, 4.0]  # replace with actual predictions
actual_ratings = [5.0, 3.0, 4.0, 4.0]  # replace with actual ratings

# Compute RMSE
rmse = np.sqrt(mean_squared_error(actual_ratings, predicted_ratings))
print(f"RMSE: {rmse}")

In [ ]:
import pickle

# Prepare the model data to be saved
model_data = {
    'U': U,
    'sigma': sigma,
    'Vt': Vt,
    'preds_df': preds_df,
    'user_ratings_mean': user_ratings_mean
}

# Save the model data to a file
with open('/content/svd_recommendation_model.pkl', 'wb') as f:
    pickle.dump(model_data, f)

print("Model components saved successfully to /content/svd_recommendation_model.pkl")

In [ ]:
# Load the saved model using pickle
with open('/content/svd_recommendation_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)